# H3 Area-Weighted Estimation

This notebook demonstrates `estimate_h3_census_indexes()` with a small synthetic finalized census layer. It avoids live downloads while preserving the same input shape used by real `census_data.gpkg` layers.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import geopandas as gpd
from shapely.geometry import box

from istat_census_data import estimate_h3_census_indexes


## Create a Minimal Finalized Census Layer


In [ ]:
source = gpd.GeoDataFrame(
    {
        "SEZ2021": ["5809100001", "5809100002"],
        "PRO_COM": ["58091", "58091"],
        "P1": [100, 200],
        "P2": [40, 80],
    },
    geometry=[
        box(12.0, 41.0, 12.01, 41.01),
        box(12.01, 41.0, 12.02, 41.01),
    ],
    crs="EPSG:4326",
)

aoi = gpd.GeoDataFrame(
    geometry=[box(12.0, 41.0, 12.02, 41.01)],
    crs="EPSG:4326",
)


The finalized package workflow writes layers named `census<year>`, such as `census2021`. The synthetic file below follows that convention.


In [ ]:
with TemporaryDirectory() as tmp_dir:
    census_data_path = Path(tmp_dir) / "census_data.gpkg"
    source.to_file(census_data_path, layer="census2021", driver="GPKG")

    h3_estimates = estimate_h3_census_indexes(
        census_data_path=census_data_path,
        year=2021,
        aoi=aoi,
        indexes=[],
        workers=1,
    )

h3_estimates[["h3_index", "year", "h3_resolution", "P1", "P2"]].head()


The additive index columns such as `P1` and `P2` are integer counts in the H3 attribute table. Density columns, when inspected, remain floating-point values.


## Validate Totals

`total_validation` stores the continuous area-weighted validation used to check geometric conservation. `integer_total_validation` stores the post-allocation check for the integer counts written in the H3 table.


In [ ]:
h3_estimates.attrs["total_validation"], h3_estimates.attrs["integer_total_validation"]


For real data, pass a finalized `census_data.gpkg` path, choose indexes from the catalogue, keep the fixed H3 resolution at level 11, and keep the default `boundary_ring_size=3` unless you are studying a specific AOI edge case.
